# HTC-AdaBins height head — full-scale training (GAMUS + DFC2019)

Runs `train_height.py --htc --use-dfc2019` for real, on Kaggle's free GPU, against enough data to actually mean something (unlike the local CPU smoke test, which used 8 GAMUS tiles / 2 DFC2019 tiles and correctly did not beat baseline).

**Before running:**
1. Kaggle notebook settings -> Accelerator -> **GPU T4 x2** (or P100). Internet must be ON (needed for the GitHub clone and the HuggingFace/GAMUS download).
2. Add a Kaggle **Secret** named `GITHUB_TOKEN`: a GitHub Personal Access Token (classic, `repo` scope is enough) for `jawaharananth/depthwizard`, since the repo is private. Add & Attach it via the notebook's Add-ons -> Secrets menu.
3. Upload your local `dfc2019_data/` folder as a private Kaggle Dataset (drag the `truth/Track3-Truth` and `rgb/Track3-RGB-1` subfolders in as-is), and attach that dataset to this notebook. Note the dataset's slug — cell 3 below expects it at `/kaggle/input/<your-dataset-slug>/`.

Everything else runs unattended.

In [ ]:
# Cell 1: clone the private repo using the GitHub token secret
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")

!rm -rf /kaggle/working/depthwizard
!git clone --depth 1 https://{token}@github.com/jawaharananth/depthwizard.git /kaggle/working/depthwizard
%cd /kaggle/working/depthwizard

In [ ]:
# Cell 2: dependencies. torch/torchvision are already on Kaggle's GPU image with CUDA wired up;
# only the project-specific packages need installing.
!pip install -q transformers h5py rasterio opencv-python-headless

In [ ]:
# Cell 3: point dfc2019_height.py at the uploaded DFC2019 Kaggle Dataset.
# EDIT the dataset slug below to match whatever you named it on upload.
import os

DFC_DATASET_SLUG = "dfc2019-track3"   # <-- change this to your uploaded dataset's slug
src = f"/kaggle/input/{DFC_DATASET_SLUG}"

os.makedirs("dfc2019_data", exist_ok=True)
if not os.path.exists("dfc2019_data/truth"):
    os.symlink(f"{src}/Track3-Truth", "dfc2019_data/truth")
if not os.path.exists("dfc2019_data/rgb"):
    os.makedirs("dfc2019_data/rgb", exist_ok=True)
    os.symlink(f"{src}/Track3-RGB-1", "dfc2019_data/rgb/Track3-RGB-1")

!python dfc2019_height.py   # sanity check: prints tile counts and one real sample

In [ ]:
# Cell 4: download a real GAMUS subset. 300 train / 60 val is a genuine step up from the
# 8/6 tiles the local smoke test had, while still finishing the download in a few minutes.
# Raise --limit further if Kaggle's GPU-hour budget and the download time allow it.
!python gamus.py download --split train --limit 300
!python gamus.py download --split val --limit 60

In [ ]:
# Cell 5: the real run. batch/accum sized for a T4's ~15GB VRAM at crop=518 with the Base
# backbone; drop --batch if you hit an OOM, or raise it on a P100/bigger card.
# 20 epochs over ~300+88 train tiles is a genuine training budget, not a smoke test --
# expect this cell to run for a few hours depending on the GPU Kaggle assigns.
!python train_height.py --htc --use-dfc2019 \
    --epochs 20 --batch 8 --accum 2 --crops-per-tile 4 --workers 2 \
    --out /kaggle/working/checkpoints

In [ ]:
# Cell 6: package the result for download. Kaggle keeps everything under /kaggle/working/
# as the notebook's output -- this just zips it so it's one file to grab.
!zip -r /kaggle/working/htc_checkpoint.zip /kaggle/working/checkpoints
print("Download htc_checkpoint.zip from this notebook's Output tab when the run finishes.")
print("It contains height_htc_best/htc_model.pt, htc_ok.json (only if it beat baseline),")
print("and training_report.json with the full per-epoch history.")

## After downloading

Copy `checkpoints/height_htc_best/` from the zip into the project's own `checkpoints/` directory locally. If `htc_ok.json` is present, `depth_model.py`'s `DepthBackbone.htc_available()` picks it up automatically and the pipeline starts using it -- if it's absent, the run did not beat baseline and nothing should be copied over (same licensing rule the plain-regression checkpoint already follows).